# Bloc 2 — v2 : comparaison élargie (ML + DL) sous une règle pré-enregistrée

**Projet Applied ML — Albert School × Mines Paris-PSL.** Suite de
[`02_model_comparison.ipynb`](02_model_comparison.ipynb) (v1).

Le professeur autorisant toute famille de ML ou de DL, la v2 compare 11
familles : régressions logistiques, EBM, forêts, SVM, XGBoost, CatBoost, réseau
de neurones avec embeddings, stacking et ensemble. Elle utilise une CV ancrée
sur l'année 1994 complète et une règle de simplicité du 1-SE. La règle a été
**pré-enregistrée avant la CV complète v2**
([`reports/bloc2_v2_decisions.md`](../reports/bloc2_v2_decisions.md)). Le
holdout 1996 avait déjà été lu en v1 : ici, il **confirme** une sélection faite
sur la CV seule.

Ce notebook **n'entraîne rien et ne calcule aucune métrique** : il relit les
résultats de `make night-v2` et les commente. Le rapport généré est
[`reports/v2/model_comparison.md`](../reports/v2/model_comparison.md). Tout
chiffre affiché ici provient d'une variable calculée.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, str(Path.cwd().parent / "src"))
import evaluate as E
import features as F
import models as M
import report as R

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)

# BLOC2_V2_CONFIG ne sert qu'à valider ce notebook sur des sorties de débogage.
cfg = F.load_config(os.environ.get("BLOC2_V2_CONFIG", F.PROJECT_ROOT / "configs" / "bloc2_v2.yaml"))
res = R.load_results(cfg)
names = E.metric_names(cfg["metrics"]["ks"])
rank_names = ["pr_auc", "roc_auc"] + [n for n in names if n.startswith("recall")]
prec_names = [n for n in names if n.startswith("precision")]
sel, dec = res["selection"], res["decision"]
ref = cfg["features"]["reference_set"]
retained, best = sel["retained_key"], M.model_key(sel["best_cv"], ref)
bars = M.BASELINES + list(cfg.get("references", {}))
candidates = R.candidate_keys(cfg)
styles = E.role_styles(candidates + bars, retained, best, bars)
highlight = list(dict.fromkeys([retained, best] + bars))
print(f"{len(candidates)} familles candidates, {len(bars)} barres, "
      f"{len(R.ablation_keys(dec))} ablations ; mode : {dec['mode']}.")

## 1. Protocole : CV ancrée

Chaque fold apprend sur **toute l'année 1994**, plus les blocs de 1995 qui
précèdent son bloc de validation. En v1 (`TimeSeriesSplit`), le premier fold
n'apprenait que sur quelques mois : la moyenne CV désavantageait les modèles
flexibles. Ici, le plus petit train est une année complète, comme au
déploiement.

In [ ]:
split = dec["holdout_split"]
print(f"Train : {split['n_train']:,} sinistres, {split['n_fraud_train']} fraudes | "
      f"holdout : {split['n_test']:,} sinistres, {split['n_fraud_test']} fraudes")
res["fold_info"]

## 2. Validation croisée (train seul)

Même étage A de nettoyage pour toutes les familles, même budget de tuning, même
splitter et même seed. Les barres servent de repère : le prior donne le niveau
du hasard, la baseline métier ce que donnent `Fault` × `BasePolicy`, et
`logreg_v1` le modèle retenu en v1.

In [ ]:
R.cv_frame(res, cfg, rank_names)

In [ ]:
R.cv_frame(res, cfg, prec_names)

Couleurs : le modèle **retenu** en bleu, le **meilleur en CV** en orange, les
**barres** en gris foncé, les autres familles en gris clair. Avec onze familles,
une couleur par famille serait illisible ; les tableaux donnent tous les
chiffres.

In [ ]:
fig = E.plot_fold_scores(res["cv_folds"], "pr_auc", candidates + bars,
                         f"PR-AUC par fold de la CV ancrée ({ref})", styles=styles)
plt.show()
oof = res["oof_scores"]
fig = E.plot_pr_curves(oof["y"], {k: oof[k] for k in highlight},
                       "Courbes PR out-of-fold (1995, folds de validation poolés)", styles=styles)
plt.show()

## 3. Règle de décision, étapes 1 et 2 (CV uniquement)

1. Candidat : meilleure PR-AUC moyenne en CV.
2. **Règle du 1-SE** : pour chaque famille plus simple, on calcule l'écart moyen
   au meilleur, d̄ (différences appariées par fold), et l'erreur-type corrigée
   par Nadeau-Bengio. La famille est éligible si d̄ ≤ SE, et on retient la plus
   simple des éligibles. Les IC à 95 % ne sont donnés que pour information.

Cette sélection est écrite dans `cv_selection.json` **avant** la lecture du
holdout.

In [ ]:
display(R.one_se_frame(res))
display(Markdown(f"**Meilleur en CV : {E.model_label(best)}. "
                 f"Retenu par la règle : {E.model_label(retained)}.**"))

## 4. Holdout 1996 : lecture unique de la v2

Chaque pipeline, refitté sur tout 1994–1995, score 1996. L'incertitude vient
d'un bootstrap stratifié apparié : mêmes rééchantillons pour tous les modèles,
IC percentiles à 95 %.

In [ ]:
R.holdout_frame(res, cfg, rank_names)

In [ ]:
R.holdout_frame(res, cfg, prec_names)

In [ ]:
hs = res["holdout_scores"]
fig = E.plot_pr_curves(hs["y"], {k: hs[k] for k in highlight}, "Courbes PR sur le holdout 1996",
                       styles=styles)
plt.show()
ap = res["holdout"].query("metric == 'pr_auc'").rename(columns={"model": "key"})
order = candidates + bars + R.ablation_keys(dec)
ap = ap.set_index("key").loc[order].reset_index()
ap["color"] = [styles.get(k, styles[retained])["color"] for k in ap["key"]]
fig = E.plot_intervals(ap, "PR-AUC (IC bootstrap à 95 %)", "PR-AUC sur le holdout 1996, tous modèles")
plt.show()

## 5. Étapes 3 et 4 : battre les trois barres, sans re-sélectionner

Le modèle retenu doit battre **chaque** barre (prior, baseline métier,
régression logistique v1) : il faut un IC bootstrap de la différence de PR-AUC
entièrement au-dessus de 0. Si une autre famille fait significativement mieux
sur 1996, on le rapporte sans changer de modèle, et il n'y aura pas de v3
choisie sur 1996.

In [ ]:
mine = res["diffs"].query("comparison == 'retenu_vs_autres'").copy()
mine["key"], mine["point"] = mine["model_b"], mine["diff"]
mine["label"] = ["retenu − " + E.model_label(k) for k in mine["model_b"]]
mine["color"] = [styles[k]["color"] for k in mine["key"]]
fig = E.plot_intervals(mine, "Différence de PR-AUC (IC bootstrap apparié à 95 %)",
                       f"PR-AUC du retenu moins chaque autre modèle\nretenu = {E.model_label(retained)}",
                       reference=0.0)
plt.show()
checks = dec["holdout_checks"]
lines = []
for b, ok in checks["beats_baselines"].items():
    row = mine.set_index("model_b").loc[b]
    lines.append(f"- {E.model_label(b)} : différence {R.signed(row['diff'])} "
                 f"{R.ci(row['ci_low'], row['ci_high'], sign=True)} → "
                 + ("battue" if ok else "**gain non démontré**"))
contra = ", ".join(E.model_label(k) for k in checks["contradicted_by"]) or "aucune"
lines.append(f"- Contradiction CV / holdout : {contra} (pas de re-sélection).")
display(Markdown("\n".join(lines)))

## 6. Ablations du modèle retenu (chiffres seulement)

Les mêmes hyperparamètres que le modèle retenu, avec une seule modification :
sans `Sex`, sur `fs_policytype`, ou avec SMOTE-NC sur le train (voir les
réserves au §8 des décisions v2). Les ablations ne changent pas la sélection ;
leur interprétation revient au bloc 4.

In [ ]:
rows = []
for key in R.ablation_keys(dec):
    row = res["diffs"].query("comparison == @key").iloc[0]
    abl = R.ablation_summary(dec, key)
    rows.append({"Ablation": key, "Modification": R.ablation_text(dec["ablations"][key]),
                 "PR-AUC CV": f"{R.num(abl['pr_auc_mean'])} ± {R.num(abl['pr_auc_std'])}",
                 "Holdout : ablation − retenu": R.signed(row["diff"]),
                 "IC 95 %": R.ci(row["ci_low"], row["ci_high"], sign=True),
                 "Verdict": R.verdict_text(row["verdict"], key, retained)})
pd.DataFrame(rows)

## 7. Conclusion (générée)

Construite à partir des variables : aucun chiffre n'est saisi à la main.

In [ ]:
h = res["holdout"].set_index(["model", "metric"])
ret_ap, v1_ap = h.loc[(retained, "pr_auc")], h.loc[("logreg_v1", "pr_auc")]
cv_best = res["cv_summary"].set_index("model").loc[best]
failed = [E.model_label(b) for b, ok in checks["beats_baselines"].items() if not ok]
txt = f'''
- **Sélection (CV du train seule)** : meilleure PR-AUC moyenne pour {E.model_label(best)}
  ({R.num(cv_best["pr_auc_mean"])} ± {R.num(cv_best["pr_auc_std"])}). Règle du 1-SE →
  **{E.model_label(retained)}**.
- **Holdout 1996** : PR-AUC du retenu {R.num(ret_ap["point"])} {R.ci(ret_ap["ci_low"], ret_ap["ci_high"])},
  contre {R.num(v1_ap["point"])} {R.ci(v1_ap["ci_low"], v1_ap["ci_high"])} pour le modèle retenu en v1.
- **Barres** : {"toutes battues" if checks["beats_all_baselines"] else "gain non démontré face à " + ", ".join(failed)}.
- **CV vs holdout** : {"aucune contradiction" if not checks["contradicted_by"] else ", ".join(E.model_label(k) for k in checks["contradicted_by"]) + " font significativement mieux sur 1996 ; rapporté sans re-sélection"}.
- **Pour le bloc 4** : scores bruts dans `{cfg["paths"]["predictions_dir"]}/`, pipeline dans
  `{cfg["paths"]["models_dir"]}/{retained}.joblib` ; calibration, seuil, slices et explications restent à traiter.
'''
display(Markdown(txt))